# Gemma 4 Study Companion v5

Notebook final para Kaggle orientado a una experiencia de estudio personal impulsada por **Gemma 4**. La app comienza con una entrevista por voz, construye un perfil del estudiante, extrae un temario desde PDF, genera lecciones por subtema, crea micro-retos de opción múltiple, produce audio y HTML, y utiliza una visualización de concepto basada en **Plotly/HTML** en lugar de Manim para mayor estabilidad [web:74][web:77][web:90].

## Principio central

Toda decisión pedagógica importante debe venir de Gemma 4: perfilado del estudiante, tono, nivel, estructura de lección, micro-retos, ejemplos, feedback y recomendación de visualización [web:74][web:77].


## Apuntes del cuaderno por imagen

La v8 agrega una vía multimodal real: puedes subir fotos de tus apuntes o libreta para que **Gemma 4 lea ese contenido y lo use para enriquecer la explicación**. Gemma 4 soporta entrada multimodal con texto e imágenes, y los modelos pequeños también soportan audio, así que esta integración sí tiene sentido técnico para el flujo de estudio [web:211][web:4][web:221].


In [ ]:
!apt-get update -y > /dev/null 2>&1
!apt-get install -y ffmpeg > /dev/null 2>&1
!pip install -q --upgrade git+https://github.com/huggingface/transformers.git
!pip install -q --upgrade accelerate bitsandbytes huggingface_hub gradio gTTS pydantic json-repair pymupdf pillow sentencepiece plotly kaleido

In [ ]:
!pip install -q --force-reinstall --no-cache-dir "numpy>=2.0" "scipy"


In [ ]:
import os
print("🔁 Reiniciando el runtime para aplicar numpy/scipy/transformers limpios...")
os.kill(os.getpid(), 9)

## 2. Configuración


In [ ]:
import os, json, re, textwrap, datetime
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Literal

import torch
import fitz
from PIL import Image, ImageDraw
from gtts import gTTS
from pydantic import BaseModel, Field
from json_repair import repair_json
from huggingface_hub import login
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig, pipeline
import plotly.graph_objects as go

ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT = ROOT / 'study_companion_v4'
ASSETS = OUT / 'assets'
MEDIA = OUT / 'media'
HTML = OUT / 'html'
DATA = OUT / 'data'
for p in [OUT, ASSETS, MEDIA, HTML, DATA]:
    p.mkdir(parents=True, exist_ok=True)

PROFILE_PATH = DATA / 'user_profile.json'
FAVORITES_PATH = DATA / 'favorites.json'
PROGRESS_PATH = DATA / 'progress.json'
LESSONS_PATH = DATA / 'lesson_memory.json'

for path, default in [
    (FAVORITES_PATH, []),
    (PROGRESS_PATH, {}),
    (LESSONS_PATH, {}),
]:
    if not path.exists():
        path.write_text(json.dumps(default, ensure_ascii=False, indent=2), encoding='utf-8')

HFTOKEN = os.environ.get('HFTOKEN')
if not HFTOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HFTOKEN = UserSecretsClient().get_secret('HFTOKEN')
    except Exception:
        HFTOKEN = None
if not HFTOKEN:
    raise RuntimeError('Configura HFTOKEN en Kaggle Add-ons > Secrets.')

login(token=HFTOKEN)
MODEL_ID = os.getenv('GEMMA_MODEL_ID', 'google/gemma-4-E4B-it')
assert torch.cuda.is_available(), 'Activa GPU en Kaggle.'

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4'
)

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HFTOKEN)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    token=HFTOKEN
).eval()

asr_pipe = pipeline('automatic-speech-recognition', model='openai/whisper-small', device=0 if torch.cuda.is_available() else -1)

print('Gemma listo:', MODEL_ID)
print('Whisper listo')

In [ ]:
def gemma_generate(system_prompt: str, user_prompt: str, max_new_tokens: int = 2600, temperature: float = 0.35) -> str:
    messages = [{
        'role': 'user',
        'content': [{'type': 'text', 'text': f'SYSTEM: {system_prompt} USER: {user_prompt}'}]
    }]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt'
    ).to(model.device, dtype=model.dtype if hasattr(model, 'dtype') else torch.bfloat16)
    input_len = inputs['input_ids'].shape[-1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
        )
    return processor.decode(output_ids[0][input_len:], skip_special_tokens=True).strip()


def extract_json_block(raw_text: str) -> dict:
    raw_text = raw_text.strip()
    raw_text = re.sub(r'^```json\s*', '', raw_text, flags=re.IGNORECASE)
    raw_text = re.sub(r'^```\s*|```$', '', raw_text)
    start = raw_text.find('{')
    end = raw_text.rfind('}')
    if start != -1 and end != -1:
        raw_text = raw_text[start:end+1]
    return json.loads(repair_json(raw_text))


def read_json(path: Path, default):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding='utf-8'))


def write_json(path: Path, data):
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')


def normalize_audio_input(audio_file):
    if audio_file is None:
        return None
    if isinstance(audio_file, str):
        return audio_file
    if hasattr(audio_file, 'name'):
        return audio_file.name
    return audio_file


def transcribe_audio_safe(audio_file):
    path = normalize_audio_input(audio_file)
    if not path:
        return ''
    try:
        result = asr_pipe(path, chunk_length_s=20, batch_size=4)
        return result.get('text', '').strip()
    except Exception as e:
        return f'[error_transcribiendo_audio] {e}'

## 3. Modelos de datos


In [ ]:
class LearnerProfile(BaseModel):
    nombre: str = 'Estudiante'
    objetivo: str
    nivel_autopercibido: Literal['principiante', 'intermedio', 'avanzado'] = 'principiante'
    estilo_preferido: str
    formato_preferido: str
    duracion_sesion: str
    puntos_de_bloqueo: List[str] = Field(default_factory=list)
    motivacion: str
    tono_preferido: str = 'amigable'
    preferencias_extra: str = ''

class Subtema(BaseModel):
    nombre: str
    prerrequisitos: List[str] = Field(default_factory=list)
    errores_comunes: List[str] = Field(default_factory=list)
    tipo_de_concepto: Literal['declarativo', 'procedimental', 'dinamico'] = 'declarativo'

class TemaCurricular(BaseModel):
    tema: str
    subtemas: List[Subtema] = Field(default_factory=list)

class Materia(BaseModel):
    materia: str
    unidad: str
    temas: List[TemaCurricular] = Field(default_factory=list)

class EjemploNivel(BaseModel):
    nivel: Literal['principiante', 'intermedio', 'avanzado']
    enunciado: str
    pista: str
    respuesta: str

class MicroReto(BaseModel):
    pregunta: str
    opciones: List[str]
    respuesta_correcta: int
    explicacion: str

class Leccion(BaseModel):
    titulo: str
    tema_padre: str
    subtema: str
    nivel_tecnico: Literal['principiante', 'intermedio', 'avanzado'] = 'principiante'
    enfoque_usuario: str = ''
    conceptos_previos: List[str] = Field(default_factory=list)
    mapa_conocimiento: List[str] = Field(default_factory=list)
    explicacion_paso_a_paso: List[str] = Field(default_factory=list)
    pausa_reflexion: str
    glosario: Dict[str, str] = Field(default_factory=dict)
    micro_retos: List[MicroReto] = Field(default_factory=list)
    ejemplos: List[EjemploNivel] = Field(default_factory=list)
    historia_base: str
    guion_audio: str
    latex_clave: List[str] = Field(default_factory=list)
    prompt_ilustracion: str
    html_prompt: str
    recomendacion_visual: Literal['imagen', 'html', 'combinada'] = 'html'
    visual_tipo: Literal['integral', 'complex', 'function', 'geometry', 'default'] = 'default'
    liked: bool = False
    version: int = 1
    feedback_historial: List[str] = Field(default_factory=list)

@dataclass
class PageBlock:
    page: int
    text: str
    image_paths: List[str] = field(default_factory=list)

## 4. Prompts de Gemma 4


In [ ]:
PROFILE_SYSTEM = r'''
Eres un diseñador de perfiles de aprendizaje. Recibes respuestas de entrevista de un estudiante.
Devuelve SOLO JSON válido con esta forma:
{
  "nombre": "...",
  "objetivo": "...",
  "nivel_autopercibido": "principiante|intermedio|avanzado",
  "estilo_preferido": "...",
  "formato_preferido": "...",
  "duracion_sesion": "...",
  "puntos_de_bloqueo": ["..."],
  "motivacion": "...",
  "tono_preferido": "...",
  "preferencias_extra": "..."
}
'''

CURRICULUM_SYSTEM = r'''
Eres un estructurador de temarios universitarios. Devuelve SOLO JSON válido con esta forma:
{
  "materia": "...",
  "unidad": "...",
  "temas": [
    {
      "tema": "...",
      "subtemas": [
        {
          "nombre": "...",
          "prerrequisitos": ["..."],
          "errores_comunes": ["..."],
          "tipo_de_concepto": "declarativo|procedimental|dinamico"
        }
      ]
    }
  ]
}
'''

LESSON_SYSTEM = r'''
Eres un tutor personal de matemáticas. Recibes un subtema, un perfil de estudiante y un enfoque adicional.
Devuelve SOLO JSON válido con esta forma:
{
  "titulo": "...",
  "conceptos_previos": ["..."],
  "mapa_conocimiento": ["..."],
  "explicacion_paso_a_paso": ["..."],
  "pausa_reflexion": "...",
  "glosario": {"termino": "definicion"},
  "micro_retos": [
    {"pregunta": "...", "opciones": ["...","...","...","..."], "respuesta_correcta": 0, "explicacion": "..."}
  ],
  "ejemplos": [
    {"nivel": "principiante", "enunciado": "...", "pista": "...", "respuesta": "..."}
  ],
  "historia_base": "...",
  "guion_audio": "...",
  "latex_clave": ["\(a+b\)", "\[x^2\]"],
  "prompt_ilustracion": "...",
  "html_prompt": "...",
  "recomendacion_visual": "imagen|html|combinada",
  "visual_tipo": "integral|complex|function|geometry|default"
}
Reglas:
- Todo debe venir de Gemma 4: perfil, micro-retos, explicación, ejemplos y recomendación visual.
- Empieza desde lo más accesible compatible con el perfil.
- Usa solo delimitadores LaTeX \( \) y \[ \].
- Los micro-retos deben ser claros, breves y con exactamente 4 opciones.
- La práctica debe ser prioritaria.
'''

FEEDBACK_SYSTEM = r'''
Eres un editor de lecciones personalizadas. Recibes una lección y feedback del usuario.
Devuelve SOLO el JSON completo actualizado. Mantén coherencia con el perfil del estudiante.
'''

## 5. Perfil y temario


In [ ]:
INTERVIEW_QUESTIONS = [
    '¿Cómo te llamas o cómo quieres que te llame la app?',
    '¿Qué materia o tema quieres dominar primero?',
    '¿Cómo describirías tu nivel actual en ese tema?',
    '¿Prefieres explicaciones intuitivas, formales, mixtas o con muchos ejemplos?',
    '¿Qué formato te ayuda más: audio, visual, práctica, lectura o mezcla?',
    '¿Cuánto tiempo quieres estudiar por sesión?',
    '¿Qué parte suele bloquearte más o frustrarte?',
    '¿Para qué quieres aprender esto en este momento?'
]


def build_profile_from_interview(interview_text: str) -> LearnerProfile:
    raw = gemma_generate(PROFILE_SYSTEM, f'Entrevista transcrita del estudiante: {interview_text}', max_new_tokens=1800)
    profile = LearnerProfile(**extract_json_block(raw))
    write_json(PROFILE_PATH, profile.model_dump())
    return profile


def load_profile() -> Optional[LearnerProfile]:
    if not PROFILE_PATH.exists():
        return None
    return LearnerProfile(**read_json(PROFILE_PATH, {}))


def extract_pdf_content(pdf_path: str) -> List[PageBlock]:
    doc = fitz.open(pdf_path)
    blocks = []
    for page_idx, page in enumerate(doc):
        text = page.get_text('text')
        pix = page.get_pixmap(matrix=fitz.Matrix(1.4, 1.4))
        img_path = ASSETS / f'page_{page_idx+1}.png'
        pix.save(str(img_path))
        blocks.append(PageBlock(page=page_idx+1, text=text, image_paths=[str(img_path)]))
    doc.close()
    return blocks


def pdf_blocks_to_text(blocks: List[PageBlock], max_chars: int = 18000) -> str:
    return ''.join([f'PÁGINA {b.page} {b.text}' for b in blocks])[:max_chars]


def build_curriculum_from_pdf_text(pdf_text: str) -> Materia:
    raw = gemma_generate(CURRICULUM_SYSTEM, f'Texto del temario: {pdf_text}', max_new_tokens=2600)
    return Materia(**extract_json_block(raw))

## 6. Lecciones, progreso y favoritos


In [ ]:
def mark_progress(subtopic: str, status: str):
    progress = read_json(PROGRESS_PATH, {})
    progress[subtopic] = {
        'status': status,
        'updated_at': datetime.datetime.now().isoformat(timespec='seconds')
    }
    write_json(PROGRESS_PATH, progress)


def load_progress_text() -> str:
    progress = read_json(PROGRESS_PATH, {})
    if not progress:
        return 'Aún no hay progreso guardado.'
    return ''.join([f"- {k}: {v.get('status', 'sin estado')} ({v.get('updated_at', '')})" for k, v in progress.items()])


def save_favorite(title: str, kind: str, path: str, subtopic: str, reason: str = 'me gustó'):
    favs = read_json(FAVORITES_PATH, [])
    favs.append({
        'title': title,
        'kind': kind,
        'path': path,
        'subtopic': subtopic,
        'reason': reason,
        'timestamp': datetime.datetime.now().isoformat(timespec='seconds')
    })
    write_json(FAVORITES_PATH, favs)


def favorites_gallery_markdown() -> str:
    favs = read_json(FAVORITES_PATH, [])
    if not favs:
        return 'No hay favoritos guardados todavía.'
    rows = []
    for i, fav in enumerate(reversed(favs), start=1):
        rows.append(f"### {i}. {fav['title']}
- Tipo: {fav['kind']}
- Subtema: {fav['subtopic']}
- Motivo: {fav['reason']}
- Ruta: {fav['path']}
- Fecha: {fav['timestamp']}")
    return ''.join(rows)


def generate_lesson(profile: LearnerProfile, tema_padre: str, subtema: Subtema, nivel: str = 'principiante', enfoque: str = '') -> Leccion:
    progress = read_json(PROGRESS_PATH, {})
    prompt = f'''Perfil del estudiante: {json.dumps(profile.model_dump(), ensure_ascii=False, indent=2)}

Tema padre: {tema_padre}
Subtema: {subtema.nombre}
Tipo de concepto: {subtema.tipo_de_concepto}
Prerrequisitos: {', '.join(subtema.prerrequisitos) or 'ninguno'}
Errores comunes: {', '.join(subtema.errores_comunes) or 'ninguno'}
Nivel técnico solicitado: {nivel}
Enfoque adicional: {enfoque or 'sin enfoque adicional'}
Estado de progreso actual: {json.dumps(progress.get(subtema.nombre, {}), ensure_ascii=False)}'''
    raw = gemma_generate(LESSON_SYSTEM, prompt, max_new_tokens=3600)
    data = extract_json_block(raw)
    lesson = Leccion(
        titulo=data['titulo'],
        tema_padre=tema_padre,
        subtema=subtema.nombre,
        nivel_tecnico=nivel,
        enfoque_usuario=enfoque,
        conceptos_previos=data.get('conceptos_previos', []),
        mapa_conocimiento=data.get('mapa_conocimiento', []),
        explicacion_paso_a_paso=data.get('explicacion_paso_a_paso', []),
        pausa_reflexion=data.get('pausa_reflexion', ''),
        glosario=data.get('glosario', {}),
        micro_retos=[MicroReto(**r) for r in data.get('micro_retos', [])],
        ejemplos=[EjemploNivel(**e) for e in data.get('ejemplos', [])],
        historia_base=data.get('historia_base', ''),
        guion_audio=data.get('guion_audio', ''),
        latex_clave=data.get('latex_clave', []),
        prompt_ilustracion=data.get('prompt_ilustracion', ''),
        html_prompt=data.get('html_prompt', ''),
        recomendacion_visual=data.get('recomendacion_visual', 'html'),
        visual_tipo=data.get('visual_tipo', 'default'),
    )
    memory = read_json(LESSONS_PATH, {})
    memory[subtema.nombre] = lesson.model_dump()
    write_json(LESSONS_PATH, memory)
    mark_progress(subtema.nombre, 'started')
    return lesson


def apply_feedback(lesson: Leccion, comment: str) -> Leccion:
    payload = {'lesson': lesson.model_dump(), 'feedback': comment}
    raw = gemma_generate(FEEDBACK_SYSTEM, json.dumps(payload, ensure_ascii=False), max_new_tokens=3200)
    data = extract_json_block(raw)
    updated = Leccion(**{**lesson.model_dump(), **data})
    updated.version = lesson.version + 1
    updated.feedback_historial = lesson.feedback_historial + [comment]
    memory = read_json(LESSONS_PATH, {})
    memory[lesson.subtema] = updated.model_dump()
    write_json(LESSONS_PATH, memory)
    return updated

## 7. Artefactos


In [ ]:
def generate_audio(lesson: Leccion) -> str:
    out = MEDIA / f'audio_{re.sub(r'[^a-zA-Z0-9]+', '_', lesson.titulo)[:40]}_v{lesson.version}.mp3'
    gTTS(text=lesson.guion_audio or lesson.historia_base, lang='es', slow=False).save(str(out))
    return str(out)


def generate_preview_image(lesson: Leccion) -> str:
    img = Image.new('RGB', (1400, 900), color=(248, 250, 252))
    draw = ImageDraw.Draw(img)
    draw.text((50, 50), lesson.titulo, fill=(17, 35, 62))
    draw.text((50, 120), f'Nivel: {lesson.nivel_tecnico}', fill=(40, 70, 100))
    draw.text((50, 180), textwrap.fill('Conceptos previos: ' + ', '.join(lesson.conceptos_previos[:6]), width=70), fill=(65, 65, 80))
    body = ''.join([f'{i+1}. {s}' for i, s in enumerate(lesson.explicacion_paso_a_paso[:5])])
    draw.text((50, 300), textwrap.fill(body, width=72), fill=(20, 20, 20))
    out = ASSETS / f'preview_{re.sub(r'[^a-zA-Z0-9]+', '_', lesson.titulo)[:40]}_v{lesson.version}.png'
    img.save(out)
    return str(out)


def build_concept_visual(lesson: Leccion) -> str:
    slug = re.sub(r'[^a-zA-Z0-9]+', '_', lesson.titulo)[:40]
    out = HTML / f'visual_{slug}_v{lesson.version}.html'
    vt = lesson.visual_tipo
    if vt == 'integral':
        x = [i/50 for i in range(0, 151)]
        y = [v**0.7 + 0.8 for v in x]
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=x, y=y, mode='lines', line=dict(color='#2563eb', width=4), name='f(x)'))
        fig.add_trace(go.Scatter(x=[0.5,0.5,2.5,2.5,0.5], y=[0,1.1,1.1,0,0], fill='toself', fillcolor='rgba(59,130,246,0.2)', line=dict(color='rgba(59,130,246,0.0)'), name='Área'))
        fig.update_layout(title=lesson.titulo, xaxis_title='x', yaxis_title='y', height=520, template='plotly_white')
    elif vt == 'complex':
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=[0,2,3], y=[0,1,3], mode='lines+markers+text', text=['0','2+i','3+3i'], textposition='top center', line=dict(color='#0f766e', width=4)))
        fig.update_layout(title=lesson.titulo, xaxis_title='Re', yaxis_title='Im', height=520, template='plotly_white', xaxis=dict(scaleanchor='y', scaleratio=1))
    elif vt == 'geometry':
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=[0,2,4,0], y=[0,3,0,0], mode='lines+markers', fill='toself', line=dict(color='#db2777', width=4)))
        fig.update_layout(title=lesson.titulo, height=520, template='plotly_white')
    else:
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=[0,1,2,3], y=[1,3,2,4], mode='lines+markers', line=dict(color='#7c3aed', width=4)))
        fig.update_layout(title=lesson.titulo, height=520, template='plotly_white')
    fig.write_html(str(out), include_plotlyjs='cdn', full_html=True, config={'displaylogo': False, 'responsive': True})
    return str(out)


def generate_practice_html(lesson: Leccion) -> str:
    out = HTML / f'{re.sub(r'[^a-zA-Z0-9]+', '_', lesson.titulo)[:40]}_v{lesson.version}.html'
    glossary_html = ''.join([f'<li><strong>{k}</strong>: {v}</li>' for k, v in lesson.glosario.items()])
    retos_html = []
    for j, r in enumerate(lesson.micro_retos, start=1):
        opts = ''.join([f'<button class="opt" data-correct="{1 if idx == r.respuesta_correcta else 0}">{chr(65+idx)}. {op}</button>' for idx, op in enumerate(r.opciones)])
        retos_html.append(f'''
        <div class="mcq" data-q="{j}">
          <p class="q"><strong>Reto {j}:</strong> {r.pregunta}</p>
          <div class="opts">{opts}</div>
          <details><summary>Ver verificación</summary><p class="mcq-exp">{r.explicacion}</p></details>
        </div>
        ''')
    examples_html = []
    for i, ex in enumerate(lesson.ejemplos, start=1):
        examples_html.append(f'''
        <details class="example-card">
          <summary>Ejemplo {i} · {ex.nivel.title()}</summary>
          <p><strong>Enunciado:</strong> {ex.enunciado}</p>
          <p><strong>Pista:</strong> {ex.pista}</p>
          <details>
            <summary>Ver respuesta</summary>
            <div class="answer">{ex.respuesta}</div>
          </details>
        </details>
        ''')
    html = f'''<!DOCTYPE html>
<html lang="es"><head>
<meta charset="utf-8" />
<meta name="viewport" content="width=device-width, initial-scale=1" />
<title>{lesson.titulo}</title>
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/katex@0.16.11/dist/katex.min.css">
<script defer src="https://cdn.jsdelivr.net/npm/katex@0.16.11/dist/katex.min.js"></script>
<script defer src="https://cdn.jsdelivr.net/npm/katex@0.16.11/dist/contrib/auto-render.min.js"
onload="renderMathInElement(document.body, {{delimiters:[{{left:'\\(', right:'\\)', display:false}},{{left:'\\[', right:'\\]', display:true}}]}});"></script>
<style>
body{{font-family:Inter,Arial,sans-serif;background:#f5f7fb;color:#17212b;margin:0;padding:32px;line-height:1.6;}}
.wrap{{max-width:1100px;margin:0 auto;background:#fff;padding:32px;border-radius:20px;box-shadow:0 10px 40px rgba(0,0,0,.08);}}
.card{{background:#f8fafc;border:1px solid #dde5f0;border-radius:16px;padding:20px;margin-top:20px;}}
.example-card,.mcq{{margin:16px 0;padding:12px 16px;background:#f9fbff;border:1px solid #d8e3f4;border-radius:14px;}}
.answer{{margin-top:10px;padding:12px;background:#eef6ea;border-radius:10px;}}
.pause{{background:#fff6db;border-left:5px solid #dfad11;padding:14px 18px;border-radius:10px;}}
.opts{{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:12px;}}
.opt{{padding:12px;border-radius:12px;border:1px solid #cdd8e7;background:white;cursor:pointer;text-align:left;}}
.opt.selected{{outline:3px solid #2b6cb0;}}
.verdict{{margin-top:12px;padding:12px;border-radius:12px;display:none;}}
.verdict.ok{{background:#e7f7e9;border:1px solid #88c48e;display:block;}}
.verdict.no{{background:#fdeaea;border:1px solid #e08b8b;display:block;}}
</style></head>
<body><div class="wrap">
<h1>{lesson.titulo}</h1>
<p><strong>Subtema:</strong> {lesson.subtema} · <strong>Nivel:</strong> {lesson.nivel_tecnico}</p>
<section class="card"><h2>Conceptos previos</h2><ul>{''.join([f'<li>{c}</li>' for c in lesson.conceptos_previos])}</ul></section>
<section class="card"><h2>Mapa de conocimiento</h2><ol>{''.join([f'<li>{m}</li>' for m in lesson.mapa_conocimiento])}</ol></section>
<section class="card"><h2>Explicación paso a paso</h2><ol>{''.join([f'<li>{s}</li>' for s in lesson.explicacion_paso_a_paso])}</ol><div class="pause"><strong>Toma una pausa:</strong> {lesson.pausa_reflexion}</div></section>
<section class="card"><h2>Glosario inicial</h2><ul>{glossary_html}</ul></section>
<section class="card"><h2>Micro-retos</h2>{''.join(retos_html)}</section>
<section class="card"><h2>Práctica por niveles</h2>{''.join(examples_html)}</section>
</div>
<script>
document.querySelectorAll('.mcq').forEach(card => {{
  const buttons = [...card.querySelectorAll('.opt')];
  const summary = document.createElement('div');
  summary.className = 'verdict';
  card.appendChild(summary);
  buttons.forEach(btn => btn.addEventListener('click', () => {{
    buttons.forEach(b => b.classList.remove('selected'));
    btn.classList.add('selected');
    const correct = btn.dataset.correct === '1';
    summary.textContent = correct ? 'Correcto. Ahora abre “Ver verificación” para revisar la explicación.' : 'Respuesta registrada. Usa “Ver verificación” para comprobarla.';
    summary.className = 'verdict ' + (correct ? 'ok' : 'no');
  }}));
}});
</script>
</body></html>'''
    out.write_text(html, encoding='utf-8')
    return str(out)

## 8. Estado y funciones de UI


In [ ]:
STATE: Dict[str, Any] = {
    'profile': load_profile(),
    'materia': None,
    'flat_topics': [],
    'lessons': {}
}


def interview_summary_text(profile: Optional[LearnerProfile]) -> str:
    if not profile:
        return 'Aún no hay perfil creado.'
    return f'''### Perfil del estudiante - Nombre: {profile.nombre} - Objetivo: {profile.objetivo} - Nivel: {profile.nivel_autopercibido} - Estilo preferido: {profile.estilo_preferido} - Formato preferido: {profile.formato_preferido} - Duración por sesión: {profile.duracion_sesion} - Bloqueos: {', '.join(profile.puntos_de_bloqueo)} - Motivación: {profile.motivacion} - Tono: {profile.tono_preferido} - Extra: {profile.preferencias_extra}'''


def build_profile_ui(audio_file, extra_text):
    voice_text = transcribe_audio_safe(audio_file) if audio_file else ''
    interview_text = (voice_text + '' if voice_text else '') + (extra_text or '')
    if not interview_text.strip():
        return 'No se recibió información de entrevista.', ''
    profile = build_profile_from_interview(interview_text)
    STATE['profile'] = profile
    return interview_summary_text(profile), interview_text


def load_pdf_ui(pdf_file):
    if pdf_file is None:
        return 'Sube un PDF.', gr.update(choices=[])
    blocks = extract_pdf_content(pdf_file.name)
    txt = pdf_blocks_to_text(blocks)
    materia = build_curriculum_from_pdf_text(txt)
    STATE['materia'] = materia
    flat = []
    for tema in materia.temas:
        for sub in tema.subtemas:
            flat.append((tema.tema, sub))
    STATE['flat_topics'] = flat
    choices = [f'{tema} :: {sub.nombre}' for tema, sub in flat]
    summary = f'Materia detectada: {materia.materia}
Unidad base: {materia.unidad}
Subtemas detectados: {len(choices)}'
    return summary, gr.update(choices=choices, value=choices[0] if choices else None)


def _selected_subtopic(selection):
    labels = [f'{tema} :: {sub.nombre}' for tema, sub in STATE['flat_topics']]
    idx = labels.index(selection)
    return STATE['flat_topics'][idx]


def generate_lesson_ui(selection, level, focus_text):
    if not STATE['profile']:
        return 'Primero crea el perfil del estudiante.', ''
    if not selection:
        return 'Selecciona un subtema.', ''
    tema, sub = _selected_subtopic(selection)
    lesson = generate_lesson(STATE['profile'], tema, sub, nivel=level, enfoque=focus_text or '')
    STATE['lessons'][selection] = lesson
    return ''.join(lesson.explicacion_paso_a_paso), json.dumps(lesson.model_dump(), ensure_ascii=False, indent=2)


def feedback_ui(selection, comment):
    if selection not in STATE['lessons']:
        return 'Genera primero una lección.', ''
    lesson = apply_feedback(STATE['lessons'][selection], comment)
    STATE['lessons'][selection] = lesson
    return ''.join(lesson.explicacion_paso_a_paso), json.dumps(lesson.model_dump(), ensure_ascii=False, indent=2)


def audio_ui(selection):
    if selection not in STATE['lessons']:
        return None
    return generate_audio(STATE['lessons'][selection])


def preview_ui(selection):
    if selection not in STATE['lessons']:
        return None
    return generate_preview_image(STATE['lessons'][selection])


def html_ui(selection):
    if selection not in STATE['lessons']:
        return None
    return generate_practice_html(STATE['lessons'][selection])


def visual_ui(selection):
    if selection not in STATE['lessons']:
        return None
    path = build_concept_visual(STATE['lessons'][selection])
    iframe = f'<iframe src="file={path}" width="100%" height="560" style="border:none;border-radius:16px;"></iframe>'
    return iframe


def favorite_ui(selection, reason):
    if selection not in STATE['lessons']:
        return favorites_gallery_markdown()
    lesson = STATE['lessons'][selection]
    html_path = str(HTML / f"{re.sub(r'[^a-zA-Z0-9]+', '_', lesson.titulo)[:40]}_v{lesson.version}.html")
    save_favorite(lesson.titulo, 'lesson', html_path, lesson.subtema, reason or 'me gustó')
    lesson.liked = True
    STATE['lessons'][selection] = lesson
    mark_progress(lesson.subtema, 'liked')
    return favorites_gallery_markdown()


def complete_ui(selection):
    if selection in STATE['lessons']:
        mark_progress(STATE['lessons'][selection].subtema, 'completed')
    return load_progress_text()

In [ ]:
# Lectura de apuntes por imagen con Gemma 4
from PIL import Image

NOTE_IMAGES_PATH = DATA / 'note_images.json'
if not NOTE_IMAGES_PATH.exists():
    write_json(NOTE_IMAGES_PATH, [])


def gemma_multimodal_generate(system_prompt: str, user_text: str, image_paths: list[str], max_new_tokens: int = 1800, temperature: float = 0.3) -> str:
    content = []
    for path in image_paths:
        content.append({'type': 'image', 'image': str(path)})
    content.append({'type': 'text', 'text': f'SYSTEM: {system_prompt} USER: {user_text}'})
    messages = [{'role': 'user', 'content': content}]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors='pt'
    ).to(model.device, dtype=model.dtype if hasattr(model, 'dtype') else torch.bfloat16)
    input_len = inputs['input_ids'].shape[-1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
        )
    return processor.decode(output_ids[0][input_len:], skip_special_tokens=True).strip()


def extract_notes_from_images(image_files) -> dict:
    if not image_files:
        return {'resumen': '', 'conceptos': [], 'dudas': [], 'texto_fuente': ''}
    paths = [img if isinstance(img, str) else img.name for img in image_files]
    system_prompt = '''
Eres un lector de apuntes manuscritos o impresos. Analiza imágenes de notas de estudio.
Devuelve SOLO JSON válido con esta forma:
{
  "resumen": "...",
  "conceptos": ["..."],
  "dudas": ["..."],
  "texto_fuente": "..."
}
Extrae lo más fiel posible y organiza conceptos y posibles dudas implícitas.
'''
    user_prompt = 'Lee estas fotos de apuntes y resume el contenido útil para estudiar. Si hay fórmulas, términos o fragmentos incompletos, interprétalos con cautela.'
    raw = gemma_multimodal_generate(system_prompt, user_prompt, paths, max_new_tokens=2200)
    data = extract_json_block(raw)
    write_json(NOTE_IMAGES_PATH, {'images': paths, 'analysis': data})
    return data


def enrich_lesson_with_notes(lesson: Leccion, notes_data: dict) -> Leccion:
    if not notes_data or not notes_data.get('texto_fuente'):
        return lesson
    payload = {
        'lesson': lesson.model_dump(),
        'notes': notes_data,
        'instruction': 'Reescribe y enriquece la lección usando los apuntes del estudiante como contexto prioritario. Mantén estructura didáctica y micro-retos.'
    }
    raw = gemma_generate(FEEDBACK_SYSTEM, json.dumps(payload, ensure_ascii=False), max_new_tokens=3200)
    data = extract_json_block(raw)
    updated = Leccion(**{**lesson.model_dump(), **data})
    updated.version = lesson.version + 1
    updated.feedback_historial = lesson.feedback_historial + ['Enriquecida con apuntes por imagen']
    memory = read_json(LESSONS_PATH, {})
    memory[lesson.subtema] = updated.model_dump()
    write_json(LESSONS_PATH, memory)
    return updated


## 9. UI con Gradio

La versión v5 debe comportarse como una app dinámica. Para eso se usa `gr.State` para guardar el índice del paso, el tema seleccionado, la lección activa y el progreso, y `@gr.render` para reconstruir la sección de la lección cuando cambian esos estados [web:161][web:162][web:123][web:135].


In [ ]:
# Helpers de depuración para generación de experiencia

def safe_status(msg: str) -> str:
    return f"### Estado {msg}"


def lesson_generation_pipeline(selection, level, focus_text):
    if not STATE.get('profile'):
        return safe_status('Primero crea el perfil del estudiante.'), ''
    if not selection:
        return safe_status('Selecciona un subtema antes de generar la experiencia.'), ''
    status_parts = ['Iniciando generación con Gemma 4...']
    result_text, lesson_json = generate_lesson_ui(selection, level, focus_text)
    status_parts.append('Lección base generada.')
    note_data = read_json(NOTE_IMAGES_PATH, {})
    if selection in STATE['lessons'] and note_data and note_data.get('analysis'):
        status_parts.append('Se detectaron apuntes; enriqueciendo lección con Gemma 4...')
        updated = enrich_lesson_with_notes(STATE['lessons'][selection], note_data['analysis'])
        STATE['lessons'][selection] = updated
        lesson_json = json.dumps(updated.model_dump(), ensure_ascii=False, indent=2)
        status_parts.append('Lección enriquecida con apuntes.')
    elif note_data and note_data.get('analysis'):
        status_parts.append('Hay apuntes cargados, pero no se pudo enlazar la lección al estado.')
    else:
        status_parts.append('No hay apuntes cargados; se usa solo perfil + temario.')
    return safe_status('- ' + '- '.join(status_parts)), lesson_json


In [ ]:
# Narración natural para audio y pasos pre-renderizados
import re


def strip_latex_for_speech(text: str) -> str:
    if not text:
        return ''
    text = re.sub(r'\$\$.*?\$\$', ' ', text, flags=re.S)
    text = re.sub(r'\$.*?\$', ' ', text, flags=re.S)
    text = re.sub(r'\[a-zA-Z]+', ' ', text)
    text = re.sub(r'[_^{}]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def build_story_audio_text(lesson: Leccion) -> str:
    sections = []
    sections.append(f"Hoy vamos a explorar {lesson.titulo}.")
    if lesson.conceptos_previos:
        sections.append("Antes de entrar al tema, conviene recordar esta idea clave: " + strip_latex_for_speech(lesson.conceptos_previos[0]))
    if lesson.explicacion_paso_a_paso:
        sections.append("Primero, piensa en esto de manera intuitiva: " + strip_latex_for_speech(lesson.explicacion_paso_a_paso[0]))
    if lesson.ejemplos:
        sections.append("Veamos ahora un ejemplo. " + strip_latex_for_speech(lesson.ejemplos[0].enunciado))
    if lesson.pausa_reflexion:
        sections.append("Para cerrar, quédate con esta reflexión: " + strip_latex_for_speech(lesson.pausa_reflexion))
    return ' '.join(sections)


def natural_audio_ui(selection):
    if not selection or selection not in STATE['lessons']:
        return None
    lesson = STATE['lessons'][selection]
    narration = build_story_audio_text(lesson)
    return tts_ui_from_text(narration)


def precompute_step_cache(selection):
    if not selection or selection not in STATE['lessons']:
        return []
    lesson = STATE['lessons'][selection]
    steps = build_step_sections(lesson)
    cached = []
    for idx, step in enumerate(steps):
        progress = int((idx + 1) / len(steps) * 100)
        note_data = read_json(NOTE_IMAGES_PATH, {})
        note_summary = ''
        if note_data and note_data.get('analysis'):
            note_summary = f"<div class='notes-box'><strong>Basado también en tus apuntes:</strong><br>{note_data['analysis'].get('resumen','')}</div>"
        html = f"<div class='block-card'><div class='block-kicker'>Paso {idx+1} de {len(steps)}</div><div class='block-title'>{step['title']}</div><div class='progress-bar-wrap'><div class='progress-bar-fill' style='width:{progress}%'></div></div><div class='block-body'>{step['content']}</div>{note_summary}</div>"
        quiz = lesson.micro_retos[min(idx, len(lesson.micro_retos)-1)] if lesson.micro_retos else None
        quiz_data = {'choices': quiz.opciones if quiz else [], 'label': quiz.pregunta if quiz else 'Micro-reto del bloque'}
        cached.append({'html': html, 'quiz': quiz_data})
    return cached


In [ ]:
# Helpers autocontenidos para v13
import re
import json


def safe_status(msg: str) -> str:
    return f"### Estado\n{msg}"


def strip_latex_for_speech(text: str) -> str:
    if not text:
        return ''
    text = re.sub(r'\$\$.*?\$\$', ' ', text, flags=re.S)
    text = re.sub(r'\$.*?\$', ' ', text, flags=re.S)
    text = re.sub(r'\\[a-zA-Z]+', ' ', text)
    text = re.sub(r'[_^{}]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def build_story_audio_text(lesson):
    sections = []
    title = getattr(lesson, 'titulo', 'este tema')
    sections.append(f"Hoy vamos a explorar {title}.")
    previos = getattr(lesson, 'conceptos_previos', []) or []
    pasos = getattr(lesson, 'explicacion_paso_a_paso', []) or []
    ejemplos = getattr(lesson, 'ejemplos', []) or []
    pausa = getattr(lesson, 'pausa_reflexion', '') or ''
    if previos:
        sections.append("Antes de entrar al tema, conviene recordar esta idea clave: " + strip_latex_for_speech(previos[0]))
    if pasos:
        sections.append("Primero, piensa en esto de manera intuitiva: " + strip_latex_for_speech(pasos[0]))
    if ejemplos:
        ejemplo = ejemplos[0]
        enunciado = ejemplo.enunciado if hasattr(ejemplo, 'enunciado') else str(ejemplo)
        sections.append("Veamos ahora un ejemplo. " + strip_latex_for_speech(enunciado))
    if pausa:
        sections.append("Para cerrar, quédate con esta reflexión: " + strip_latex_for_speech(pausa))
    return ' '.join(sections)


def natural_audio_ui(selection):
    if not selection or selection not in STATE['lessons']:
        return None
    lesson = STATE['lessons'][selection]
    narration = build_story_audio_text(lesson)
    if 'tts_ui_from_text' in globals():
        return tts_ui_from_text(narration)
    if 'audio_ui' in globals():
        return audio_ui(selection)
    return None


def lesson_generation_pipeline_impl(selection, level, focus_text):
    if not STATE.get('profile'):
        return safe_status('Primero crea el perfil del estudiante.'), ''
    if not selection:
        return safe_status('Selecciona un subtema antes de generar la experiencia.'), ''
    status_parts = ['Iniciando generación con Gemma 4...']
    result_text, lesson_json = generate_lesson_ui(selection, level, focus_text)
    status_parts.append('Lección base generada.')
    note_data = read_json(NOTE_IMAGES_PATH, {}) if 'NOTE_IMAGES_PATH' in globals() else {}
    if selection in STATE.get('lessons', {}) and note_data and note_data.get('analysis'):
        status_parts.append('Se detectaron apuntes; enriqueciendo lección con Gemma 4...')
        updated = enrich_lesson_with_notes(STATE['lessons'][selection], note_data['analysis'])
        STATE['lessons'][selection] = updated
        lesson_json = json.dumps(updated.model_dump(), ensure_ascii=False, indent=2)
        status_parts.append('Lección enriquecida con apuntes.')
    else:
        status_parts.append('No hay apuntes cargados; se usa perfil + temario.')
    return safe_status('\n- ' + '\n- '.join(status_parts)), lesson_json


def precompute_step_cache(selection):
    if not selection or selection not in STATE.get('lessons', {}):
        return []
    lesson = STATE['lessons'][selection]
    steps = build_step_sections(lesson)
    cached = []
    note_data = read_json(NOTE_IMAGES_PATH, {}) if 'NOTE_IMAGES_PATH' in globals() else {}
    note_summary = ''
    if note_data and note_data.get('analysis'):
        note_summary = f"<div class='notes-box'><strong>Basado también en tus apuntes:</strong><br>{note_data['analysis'].get('resumen','')}</div>"
    for idx, step in enumerate(steps):
        progress = int((idx + 1) / len(steps) * 100)
        html = f"<div class='block-card'><div class='block-kicker'>Paso {idx+1} de {len(steps)}</div><div class='block-title'>{step['title']}</div><div class='progress-bar-wrap'><div class='progress-bar-fill' style='width:{progress}%'></div></div><div class='block-body'>{step['content']}</div>{note_summary}</div>"
        lesson_quizzes = getattr(lesson, 'micro_retos', []) or []
        quiz = lesson_quizzes[min(idx, len(lesson_quizzes)-1)] if lesson_quizzes else None
        quiz_data = {'choices': quiz.opciones if quiz else [], 'label': quiz.pregunta if quiz else 'Micro-reto del bloque'}
        cached.append({'html': html, 'quiz': quiz_data})
    return cached


In [ ]:
# Secuencias renderizadas por Gemma 4

def build_step_sections(lesson: Leccion):
    steps = []
    steps.append({'kind': 'tema', 'title': f'Tema: {lesson.titulo}', 'content': lesson.titulo, 'progress': 10})
    if lesson.conceptos_previos:
        steps.append({'kind': 'definicion', 'title': 'Definición', 'content': lesson.conceptos_previos[0], 'progress': 20})
    if lesson.mapa_conocimiento:
        steps.append({'kind': 'mapa', 'title': 'Mapa conceptual', 'content': '<br>'.join(lesson.mapa_conocimiento[:2]), 'progress': 30})
    if lesson.explicacion_paso_a_paso:
        steps.append({'kind': 'ejemplo', 'title': 'Ejemplo guiado', 'content': lesson.explicacion_paso_a_paso[0], 'progress': 45})
    if lesson.micro_retos:
        steps.append({'kind': 'reto', 'title': 'Micro-reto', 'content': lesson.micro_retos[0].pregunta, 'progress': 60})
    if lesson.ejemplos:
        steps.append({'kind': 'practica', 'title': 'Práctica', 'content': lesson.ejemplos[0].enunciado, 'progress': 75})
    steps.append({'kind': 'cierre', 'title': 'Cierre', 'content': lesson.pausa_reflexion, 'progress': 100})
    return steps


def render_step_card(step: dict, lesson: Leccion, idx: int, total: int) -> str:
    action_audio = f'data-step="{idx}" data-action="audio"'
    action_html = f'data-step="{idx}" data-action="html"'
    action_visual = f'data-step="{idx}" data-action="visual"'
    action_next = f'data-step="{idx}" data-action="next"'
    return f'''
    <section class="lesson-card" data-kind="{step['kind']}">
      <div class="lesson-card-head">
        <div>
          <p class="eyebrow">Paso {idx+1} de {total}</p>
          <h3>{step['title']}</h3>
        </div>
        <div class="progress-pill">{step['progress']}%</div>
      </div>
      <div class="lesson-content">{step['content']}</div>
      <div class="lesson-actions">
        <button class="mini-btn" {action_audio}>Generar audio</button>
        <button class="mini-btn" {action_html}>Generar HTML</button>
        <button class="mini-btn" {action_visual}>Generar gráfica/imagen</button>
        <button class="mini-btn primary" {action_next}>{"Siguiente" if idx < total-1 else "Finalizar"}</button>
      </div>
      <div class="quiz-box">
        <p><strong>Pregunta del bloque:</strong> {lesson.micro_retos[0].pregunta if lesson.micro_retos else ''}</p>
        <p class="quiz-note">El bloque se verifica con opción múltiple y Gemma mantiene la explicación de la respuesta.</p>
      </div>
    </section>
    '''


def lesson_steps_html(lesson: Leccion) -> str:
    steps = build_step_sections(lesson)
    cards = [render_step_card(step, lesson, idx, len(steps)) for idx, step in enumerate(steps)]
    return '<div class="lesson-steps">' + ''.join(cards) + '</div>'


In [ ]:
import gradio as gr

custom_css = """
:root {
  --gemma-bg: #081018;
  --gemma-surface: #0f1a24;
  --gemma-surface-2: #132230;
  --gemma-border: rgba(127, 204, 180, .18);
  --gemma-text: #eef6f4;
  --gemma-muted: #9fb3b0;
  --gemma-primary: #2fd1a6;
  --gemma-secondary: #4aa3ff;
}
body { background: linear-gradient(180deg, #071018 0%, #09141d 45%, #0b1620 100%) !important; color: var(--gemma-text) !important; }
.gradio-container { background: transparent !important; }
.gemma-chip { display:inline-flex; align-items:center; gap:.4rem; padding:.35rem .7rem; border-radius:999px; background: rgba(47,209,166,.12); color: var(--gemma-primary); border: 1px solid rgba(47,209,166,.22); font-size: .85rem; }
.gemma-hero { padding: 1.2rem 1.4rem; border-radius: 22px; background: radial-gradient(circle at top right, rgba(74,163,255,.18), transparent 35%), linear-gradient(135deg, rgba(47,209,166,.16), rgba(19,34,48,.96)); border: 1px solid rgba(95,230,192,.16); margin-bottom: 1rem; }
.gemma-title { font-size: 2rem; font-weight: 800; letter-spacing: -0.03em; margin: 0; }
.gemma-sub { color: var(--gemma-muted); margin-top: .35rem; }
.panel { background: var(--gemma-surface); border: 1px solid var(--gemma-border); border-radius: 20px; padding: 1rem; box-shadow: 0 14px 40px rgba(0,0,0,.28); }
.progress-bar-wrap { width: 100%; background: rgba(255,255,255,.06); border-radius: 999px; overflow: hidden; border: 1px solid rgba(255,255,255,.08); margin: .6rem 0 1rem; }
.progress-bar-fill { height: 12px; background: linear-gradient(90deg, var(--gemma-primary), var(--gemma-secondary)); }
.block-card { background: linear-gradient(180deg, rgba(255,255,255,.025), rgba(255,255,255,.01)); border: 1px solid rgba(127,204,180,.16); border-radius: 18px; padding: 1.05rem 1.1rem; margin-top: .9rem; }
.block-kicker { color: var(--gemma-primary); font-size: .82rem; text-transform: uppercase; letter-spacing: .08em; }
.block-title { margin: .2rem 0 .5rem; font-size: 1.35rem; font-weight: 800; }
.block-body { color: #dce6e4; line-height: 1.7; }
.notes-box { background: rgba(74,163,255,.08); border: 1px solid rgba(74,163,255,.18); border-radius: 16px; padding: .9rem; margin-top: .8rem; }
.lesson-stack { display: grid; gap: 1rem; margin-top: 1rem; }
"""

current_selection_box = gr.Textbox(value='', visible=False)


def save_selection(sel):
    return sel or ''


def render_all_steps_html(selection):
    if not selection or selection not in STATE.get('lessons', {}):
        return 'La experiencia aparecerá aquí cuando generes una lección.', gr.update(choices=[], label='Micro-reto final', value=None)
    lesson = STATE['lessons'][selection]
    steps = build_step_sections(lesson)
    note_data = read_json(NOTE_IMAGES_PATH, {}) if 'NOTE_IMAGES_PATH' in globals() else {}
    note_summary = ''
    if note_data and note_data.get('analysis'):
        note_summary = f"<div class='notes-box'><strong>Basado también en tus apuntes:</strong><br>{note_data['analysis'].get('resumen','')}</div>"
    cards = []
    for idx, step in enumerate(steps):
        progress = int((idx + 1) / len(steps) * 100)
        cards.append(
            f"<div class='block-card'><div class='block-kicker'>Paso {idx+1} de {len(steps)}</div><div class='block-title'>{step['title']}</div><div class='progress-bar-wrap'><div class='progress-bar-fill' style='width:{progress}%'></div></div><div class='block-body'>{step['content']}</div>{note_summary}</div>"
        )
    quiz = lesson.micro_retos[-1] if getattr(lesson, 'micro_retos', []) else None
    quiz_update = gr.update(choices=quiz.opciones if quiz else [], label=quiz.pregunta if quiz else 'Micro-reto final', value=None)
    return "<div class='lesson-stack'>" + ''.join(cards) + "</div>", quiz_update


def lesson_generation_pipeline(selection, level, focus_text):
    status_md, _ = lesson_generation_pipeline_impl(selection, level, focus_text)
    html, quiz_update = render_all_steps_html(selection)
    return status_md, html, quiz_update, selection or ''


def verify_quiz_answer(selection, answer):
    if not selection or selection not in STATE['lessons']:
        return 'Primero genera una lección.'
    lesson = STATE['lessons'][selection]
    if not lesson.micro_retos:
        return 'No hay micro-reto disponible.'
    quiz = lesson.micro_retos[-1]
    chosen = quiz.opciones.index(answer) if answer in quiz.opciones else -1
    if chosen == quiz.respuesta_correcta:
        return f'✅ Correcto. {quiz.explicacion}'
    return f'❌ Incorrecto. {quiz.explicacion}'


def notes_ui(image_files):
    if not image_files:
        return 'No se cargaron imágenes.', '', gr.update(open=True)
    data = extract_notes_from_images(image_files)
    summary = f"### Apuntes detectados\n- Resumen: {data.get('resumen', '')}\n- Conceptos: {', '.join(data.get('conceptos', []))}\n- Dudas: {', '.join(data.get('dudas', []))}"
    return summary, json.dumps(data, ensure_ascii=False, indent=2), gr.update(open=False)


def safe_preview(selection):
    if not selection:
        return None
    return preview_ui(selection)


def safe_audio(selection):
    if not selection:
        return None
    return natural_audio_ui(selection)


def safe_html(selection):
    if not selection:
        return None
    return html_ui(selection)


def safe_visual(selection):
    if not selection:
        return '<div class="notes-box">Primero genera una lección.</div>'
    return visual_ui(selection)

with gr.Blocks(theme=gr.themes.Base(), css=custom_css) as demo:
    gr.HTML('<div class="gemma-hero"><span class="gemma-chip">Gemma 4 • Study Companion</span><h1 class="gemma-title">Gemma 4 Study Companion v15</h1><p class="gemma-sub">Versión corregida: audio, HTML y preview ya no dependen de gr.State para evitar errores de tipo stateful.</p></div>')

    with gr.Row():
        with gr.Column(scale=1):
            with gr.Accordion('1. Perfil del estudiante', open=True) as acc_profile:
                interview_audio = gr.Audio(label='Entrevista por voz', sources=['microphone', 'upload'], type='filepath', format='wav')
                interview_text = gr.Textbox(label='Notas adicionales', lines=4)
                profile_btn = gr.Button('Crear perfil', variant='primary')
                profile_md = gr.Markdown(value=interview_summary_text(STATE['profile']))
                interview_transcript = gr.Textbox(label='Transcripción', lines=5)
            with gr.Accordion('2. Temario y subtema', open=True) as acc_curriculum:
                pdf_in = gr.File(label='Temario en PDF', file_types=['.pdf'])
                load_btn = gr.Button('Extraer temario')
                summary_md = gr.Markdown()
                topic_dd = gr.Dropdown(label='Subtema detectado', choices=[])
                level_dd = gr.Dropdown(label='Nivel técnico', choices=['principiante', 'intermedio', 'avanzado'], value='principiante')
                focus_text = gr.Textbox(label='Foco de la lección', lines=3)
            with gr.Accordion('3. Tus apuntes', open=True) as acc_notes:
                notes_images = gr.File(label='Fotos de la libreta / apuntes', file_count='multiple', file_types=['image'])
                notes_btn = gr.Button('Leer apuntes con Gemma')
                notes_md = gr.Markdown('Sube imágenes de tus apuntes para enriquecer la lección antes de generarla.')
                notes_json = gr.Code(label='Análisis de apuntes', language='json')
            with gr.Accordion('4. Generación', open=True) as acc_generate:
                lesson_btn = gr.Button('Generar experiencia', variant='primary')
                feedback_box = gr.Textbox(label='Feedback a Gemma', lines=3)
                feedback_btn = gr.Button('Aplicar feedback')
                progress_md = gr.Markdown(value=load_progress_text())
                current_selection_hidden = gr.Textbox(value='', visible=False)
        with gr.Column(scale=2):
            with gr.Group(elem_classes=['panel']):
                generation_status = gr.Markdown('### Estado\nEsperando configuración.')
                lesson_view = gr.HTML('La experiencia aparecerá aquí cuando generes una lección.')
                with gr.Row():
                    preview_btn = gr.Button('Preview')
                    audio_btn = gr.Button('Audio narrado')
                    html_btn = gr.Button('HTML')
                    visual_btn = gr.Button('Gráfica/imagen')
                out_img = gr.Image(label='Preview visual')
                out_audio = gr.Audio(label='Narración natural', type='filepath')
                out_html = gr.File(label='HTML de práctica')
                out_visual = gr.HTML(label='Visualización del concepto')
                quiz_choice = gr.Radio([], label='Micro-reto final')
                quiz_check = gr.Button('Verificar respuesta')
                quiz_result = gr.Markdown('')

    profile_btn.click(build_profile_ui, inputs=[interview_audio, interview_text], outputs=[profile_md, interview_transcript]).then(lambda: gr.update(open=False), outputs=acc_profile)
    load_btn.click(load_pdf_ui, inputs=pdf_in, outputs=[summary_md, topic_dd]).then(lambda: gr.update(open=False), outputs=acc_curriculum)
    topic_dd.change(save_selection, inputs=topic_dd, outputs=current_selection_hidden)
    notes_btn.click(notes_ui, inputs=notes_images, outputs=[notes_md, notes_json, acc_notes])
    lesson_btn.click(lesson_generation_pipeline, inputs=[topic_dd, level_dd, focus_text], outputs=[generation_status, lesson_view, quiz_choice, current_selection_hidden]).then(lambda: gr.update(open=False), outputs=acc_generate)
    feedback_btn.click(lesson_generation_pipeline, inputs=[topic_dd, level_dd, feedback_box], outputs=[generation_status, lesson_view, quiz_choice, current_selection_hidden])
    quiz_check.click(verify_quiz_answer, inputs=[current_selection_hidden, quiz_choice], outputs=quiz_result)
    preview_btn.click(safe_preview, inputs=current_selection_hidden, outputs=out_img)
    audio_btn.click(safe_audio, inputs=current_selection_hidden, outputs=out_audio)
    html_btn.click(safe_html, inputs=current_selection_hidden, outputs=out_html)
    visual_btn.click(safe_visual, inputs=current_selection_hidden, outputs=out_visual)


demo.launch(inline=True, share=True, debug=True)


## 11. Notas finales

- Gemma 4 sigue siendo la capa central de decisión y generación [web:74][web:77].
- El audio se procesa por ruta de archivo con `gr.Audio(..., type='filepath')`, útil para ASR con Whisper [web:82][web:68].
- La visualización conceptual usa Plotly HTML exportable e interactivo [web:90][web:98][web:100].
- Si quieres historial real entre sesiones, analíticas de aprendizaje o múltiples usuarios, sí conviene pasar a una base de datos o almacenamiento persistente; para Kaggle puede bastar `json` local, pero para una versión más seria yo usaría SQLite o una base externa [web:117][web:110].
